<h2>Food (v0.03) - 'Food (v0.03).ipynb'</h2>

<u><h5>Context</h5></u>

The notebook aims to isolate NZF impacts modelled for individual commodities with 'CPC' codes forwarded over by David Forengie on 23rd September 2026.

The objectives of the notebook are:

<ol>
    <li>
        To develop the representation of CPC codes provided by David, as its not completely clear how these work (e.g. where some CPC codes are only 3 digits, whereas others are 5). 
    </li>
    <li>
        To read-in 'v0.64' of the 'Impact' analysis results for the country in question, then isolate these records for the CPC codes identified in the step above.
    </li>
</ol>

<u><h5>Version Documentation</h5></u>

'v0.01' outlined an initial method for completing the process. 'v0.02' refined the method and developed a fully functioning notebook, utilising results from the 'Impact' notebooks which now include CPC codes as fields in the output data tables. Upon inspection, it was found that the identification of CPC codes wasn't functioning properly, where CPC codes specified by David wouldn't show in the dataset, yet HSa4 codes given in the CPC manual would show. This likely explains much of the losses observed when pulling by CPC codes, so 'v0.03' utilises a new version of the codes sent through David which have been mapped to the HS system ahead of time. 

In [1]:
# Import packages
import os
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def download_link(df, title="Download CSV file", filename="data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.03"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/food/"

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "328"
iso_country = "Guyana"
year = 2025
gdp = 27.1000e9

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

## 1.2 Import and Format 'Impact Analysis' Results

In [3]:
impact_im_c = [
    "trade_idx", "reporterCode", "reporterDesc", "partnerCode", "partnerDesc", "flowDesc", "motDesc", 
    "HS6", "HS6_desc", "HS4", "HS4_desc", "HS2", "HS2_desc", "cpc", "cpc_desc", "eora_desc", "gtap", "gtap_desc", 
    "cifvalue", "fobvalue", "primaryValue", "qtyUnitAbbr", "qty", "netWgt", "vol_kg_vtw", 
    "sea_dist_km", "sea_dist_nm", "tnm", "fac_vessel_types", "fac_vessel_types_str", 
    #"TandT_match_desc", "TandT_ad_val_pct", "TandT_N", "TandT_ciffob", "TandT_cifvalue", "TandT_fobvalue", "cost_int_match_desc",
    "bau_usd_26", "levy_usd_26", "asis_usd_26", "brazil_usd_26", "liscr_usd_26", "direct_usd_26", 
    "bau_usd_29", "levy_usd_29", "asis_usd_29", "brazil_usd_29", "liscr_usd_29", "direct_usd_29", 
    "bau_usd_32", "levy_usd_32", "asis_usd_32", "brazil_usd_32", "liscr_usd_32", "direct_usd_32", 
    "bau_usd_35", "levy_usd_35", "asis_usd_35", "brazil_usd_35", "liscr_usd_35", "direct_usd_35", 
    "bau_usd_38", "levy_usd_38", "asis_usd_38", "brazil_usd_38", "liscr_usd_38", "direct_usd_38", 
    "bau_usd_41", "levy_usd_41", "asis_usd_41", "brazil_usd_41", "liscr_usd_41", "direct_usd_41", 
    "bau_usd_44", "levy_usd_44", "asis_usd_44", "brazil_usd_44", "liscr_usd_44", "direct_usd_44", 
    "bau_usd_47", "levy_usd_47", "asis_usd_47", "brazil_usd_47", "liscr_usd_47", "direct_usd_47", 
    "bau_usd_50", "levy_usd_50", "asis_usd_50", "brazil_usd_50", "liscr_usd_50", "direct_usd_50"	
]

impact_im_d = {
    "reporterCode":"str", "partnerCode":"str", 
    "HS6":"str", "HS4":"str", "HS2":"str", "HS2":"str", "gtap":"str", "cpc":"str"
}

impact_im = pd.read_csv(
    input_dir + "impact/{0}/impact_im_v0.64.csv".format(iso_country), 
    dtype=impact_im_d,
    usecols=impact_im_c
)[impact_im_c]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(impact_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(impact_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(impact_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(impact_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(impact_im.vol_kg_vtw.sum() / 1e6, 1)))

impact_im["cpc_cl"] = [str(i).split('.')[0] for i in impact_im.cpc.values]
impact_im = impact_im.drop(columns="trade_idx").reset_index(drop=True).reset_index().rename(columns={"index":"trade_idx"})
display(impact_im.head(2))

Aggregate trade portfolio statistics for 'Guyana': 

Number of trade records present in dataset: 21216
Number of partner countries present in dataset: 140
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$9565.0m
Total weight within the dataset: 8006.2 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl
0,0,328,Guyana,056,Belgium,Import,Sea,320820,320820 | Paints and varnishes,3208,"3208 | Paints, varnishes; (",32,32 | Tanning or dyeing ex,3511,Paints and varnishes (including enamels and la...,"Petroleum, Chemical and Non-Metallic Mineral P...",33,"CRP - Chemical, rubber, plastic products",1685.663,NaN,1685.663,kg,60.0,60.0,325.332959,6364.0,3954.405044,1286.498294,['Container'],['Container'],6.724307,6.712566,6.713713,6.713713,6.718975,6.718975,7.033967,12.466396,7.758691,7.203223,7.222785,7.403686,6.887489,12.208926,8.166511,7.480402,7.308742,7.639755,6.759021,13.276379,10.126866,8.856600,8.194031,7.962888,6.632932,14.116045,11.615884,10.885069,8.772765,8.534223,6.509224,14.304634,13.666812,13.784320,9.510589,8.964926,6.383036,14.010618,13.426464,14.710664,10.881377,9.129239,6.264158,12.497811,13.026046,13.649937,10.232424,9.237106,6.142940,11.480307,11.670033,14.556860,12.026009,9.298275,3511
1,1,328,Guyana,056,Belgium,Import,Sea,560750,"560750 | Twine, cordage, rope",5607,"5607 | Twine, cordage, rope",56,"56 | Wadding, felt and no",2731,"Twine, cordage, rope and cables",Textiles and Wearing Apparel,27,TEX - Textiles,4954.592,NaN,4954.592,kg,1267.0,1267.0,1268.871011,6364.0,3954.405044,5017.629927,['Container'],['Container'],26.226294,26.180503,26.184978,26.184978,26.205498,26.205498,27.434038,48.621722,30.260623,28.094172,28.170470,28.876023,26.862741,47.617531,31.851214,29.175236,28.505722,29.796748,26.361686,51.780837,39.497034,34.542712,31.958547,31.057036,25.869914,55.055720,45.304534,42.454194,34.215737,33.285370,25.387425,55.791260,53.303612,53.761921,37.093414,34.965209,24.895264,54.644530,52.366201,57.374868,42.439794,35.606067,24.431613,48.744246,50.804480,53.237793,39.908733,36.026771,23.958836,44.775756,45.515728,56.774996,46.904113,36.265343,2731


# 2. Subset the 'Impact Analysis' results for food security-related CPC codes

## 2.1 Let's get a sense of how much of import value is associated with food-relevant goods across classification systems

### 2.1.1 Harmonised System (HS01-23)

In [4]:
#print("HS groupings: \n\n{0}\n".format(pd.Series(impact_im.HS2_desc.unique()).sort_values().values))
HSa2_good_groups = [
    "01", "02", "03", "04", "05", "07", "08", # "06", 
    "09", "10", "11", "12", "13", "14", "15", "16", 
    "17", "18", "19", "20", "21", "22", "23"
]

impact_im_HSa2_food = impact_im[(impact_im.HS2.isin(HSa2_good_groups))]
impact_im_HSa2_food_g = impact_im_HSa2_food.groupby(["HS2_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_HSa2_food_g["pct_val"] = round(100 * impact_im_HSa2_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_HSa2_food_g["pct_vol"] = round(100 * impact_im_HSa2_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_HSa2_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_HSa2_food_g.round(1)
))

Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): $615.6m (6.4%), ... 

                              cifvalue   vol_kg_vtw  pct_val  pct_vol
HS2_desc                                                             
22 | Beverages, spirits a  114401111.7  414063139.6      1.2      5.2
19 | Preparations of cere   83059303.7   29765384.9      0.9      0.4
04 | Dairy produce; birds   77827122.2   27400996.3      0.8      0.3
21 | Miscellaneous edible   56878672.6   17413926.4      0.6      0.2
20 | Preparations of vege   48001956.4   34616743.6      0.5      0.4
07 | Vegetables and certa   38679486.1   69148796.4      0.4      0.9
10 | Cereals                38300516.4  145322818.8      0.4      1.8
15 | Animal, vegetable or   27719927.2   23409183.9      0.3      0.3
16 | Meat, fish, crustace   24912739.0    5609759.5      0.3      0.1
17 | Sugars and sugar con   19877102.5   27646321.0      0.2      0.3
23 | Food industries, res   17907486.4   18542083.5      0.2

### 2.1.2 EORA groupings ('Food & Beverages' and 'Fishing')

In [5]:
print("EORA groupings: \n\n{0}\n".format(pd.Series(impact_im.eora_desc.unique()).sort_values().values))
eora_good_groups = ["Food & Beverages", "Agriculture", "Fishing"]

impact_im_eora_food = impact_im[(impact_im.eora_desc.isin(eora_good_groups))]
impact_im_eora_food_g = impact_im_eora_food.groupby(["eora_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_eora_food_g["pct_val"] = round(100 * impact_im_eora_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_eora_food_g["pct_vol"] = round(100 * impact_im_eora_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_eora_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_eora_food_g.round(1)
))

EORA groupings: 

['Agriculture' 'Electrical and Machinery' 'Fishing' 'Food & Beverages'
 'Metal Products' 'Mining and Quarrying' 'Other Manufacturing'
 'Petroleum, Chemical and Non-Metallic Mineral Products'
 'Textiles and Wearing Apparel' 'Transport Equipment' 'Wood and Paper' nan]

Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): $540.3m (5.6%), ... 

                     cifvalue   vol_kg_vtw  pct_val  pct_vol
eora_desc                                                   
Food & Beverages  340643230.9  505012881.1      3.6      6.3
Agriculture       199595018.6  233702486.8      2.1      2.9
Fishing              102032.7      15220.8      0.0      0.0



### 2.1.3 GTAP (16x headings)

In [6]:
#print("GTAP groupings: \n\n{0}\n".format(pd.Series(impact_im.gtap_desc.unique()).sort_values().values))
gtap_food_groups = [
    "GRO - Cereal grains n.e.c.", "PDR - Paddy rice", "PCR - Processed rice", "SGR - Sugar", 
    "CMT - Bovine meat prods", "CTL - Bovine cattle, sheep and goats, horses", "FSH - Fishing", "OMT - Meat products n.e.c.",
    "MIL - Dairy products", "OFD - Food products n.e.c.", "V_F - Vegetables, fruit, nuts", "OSD - Oil seeds", 
    "OCR - Crops n.e.c.", "PFB - Plant-based fibers", "VOL - Vegetable oils and fats", 
    "OAP - Animal products n.e.c."
]

impact_im_gtap_food = impact_im[(impact_im.gtap_desc.isin(gtap_food_groups))]
impact_im_gtap_food_g = impact_im_gtap_food.groupby(["gtap_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_gtap_food_g["pct_val"] = round(100 * impact_im_gtap_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_gtap_food_g["pct_vol"] = round(100 * impact_im_gtap_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via GTAP: ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_gtap_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_gtap_food_g.head(15).round(1)
))

Value of food imports identified via GTAP: $456.6m (4.8%), ... 

                                  cifvalue   vol_kg_vtw  pct_val  pct_vol
gtap_desc                                                                
OFD - Food products n.e.c.     238461602.5  116801712.2      2.5      1.5
MIL - Dairy products            69891325.6   22671364.7      0.7      0.3
V_F - Vegetables, fruit, nuts   46582877.3   75187779.7      0.5      0.9
VOL - Vegetable oils and fats   40752363.3   47795588.5      0.4      0.6
OMT - Meat products n.e.c.      23111639.6    6098051.6      0.2      0.1
GRO - Cereal grains n.e.c.      15473280.2   63935257.3      0.2      0.8
SGR - Sugar                     11203569.1   24574720.6      0.1      0.3
CMT - Bovine meat prods          4998865.3    1303971.5      0.1      0.0
OCR - Crops n.e.c.               4878236.3    1513803.6      0.1      0.0
PCR - Processed rice              383449.1     666051.1      0.0      0.0
OSD - Oil seeds                   333110.5     

## 2.2 Comparison of results

In [7]:
print("Share of total import value associated with food-relevant goods by classification system...\n\nHSa2 {0}% \nGTAP {1}% \nEORA {2}%".format(
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))

Share of total import value associated with food-relevant goods by classification system...

HSa2 6.4% 
GTAP 4.8% 
EORA 5.6%


# 3. Take HSa4 codes associated with specific CPCs

## 3.1 Set HSa4 Codes we're interested in

In [8]:
hsa4_codes_food = {
    "Dairy": [
        "0401", "0402", "0403", "0404", "0405", "0406",
    ],
    "Fats and Oils": [
        "1501", "1502", "1503", "1504", "1505", "1506", "1507", "1508", "1509", "1510"
        "1511", "1512", "1513", "1514", "1515", "1516", "1517"
    ],
    "Fruits": [
        "0709", "0801", "0802", "0803", "0804", "0805", "0806", "0807", "0808", "0809", 
        "0810", "0813", "2009"
    ],
    "Meat and Prep": [
        "0201", "0203", "0204", "0205", "0207", "0208", "0209", "2301"
    ],
    "Processed Cereals": [
        "1006", "1101", "1102", "1104", "1105", "1106", "1107", "1108", "1109", "1302",
        "1602", "1901", "1902", "1903", "1904", "1905", "2005", "2007", "2104", "2106", 
        "3505"
    ],
    "Sugar and Honey": [
        "1701", "1702", "1703", "1704", "2006"
    ],
    "Unprocessed Cereals": [
        "1001", "1002", "1003", "1004", "1005", "1007", "1008"
    ],
    "Vegetables": [
        "0701", "0702", "0703", "0704", "0705", "0706", "0707", "0708", "0709", "0710",
        "0712", "0713", "0714", "1209", "2009"
    ]
}

## 3.2 Identify 'Food Security' groups of interest via CPC

In [9]:
impact_im["food_group"] = np.nan

for food_group in list(hsa4_codes_food.keys()):
    impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group

display(impact_im.head(2))

/var/folders/_k/81z8ryjd28lc4f0bb4dm7gkw0000gn/T/ipykernel_16136/1412186970.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Dairy' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl,food_group
0,0,328,Guyana,056,Belgium,Import,Sea,320820,320820 | Paints and varnishes,3208,"3208 | Paints, varnishes; (",32,32 | Tanning or dyeing ex,3511,Paints and varnishes (including enamels and la...,"Petroleum, Chemical and Non-Metallic Mineral P...",33,"CRP - Chemical, rubber, plastic products",1685.663,NaN,1685.663,kg,60.0,60.0,325.332959,6364.0,3954.405044,1286.498294,['Container'],['Container'],6.724307,6.712566,6.713713,6.713713,6.718975,6.718975,7.033967,12.466396,7.758691,7.203223,7.222785,7.403686,6.887489,12.208926,8.166511,7.480402,7.308742,7.639755,6.759021,13.276379,10.126866,8.856600,8.194031,7.962888,6.632932,14.116045,11.615884,10.885069,8.772765,8.534223,6.509224,14.304634,13.666812,13.784320,9.510589,8.964926,6.383036,14.010618,13.426464,14.710664,10.881377,9.129239,6.264158,12.497811,13.026046,13.649937,10.232424,9.237106,6.142940,11.480307,11.670033,14.556860,12.026009,9.298275,3511,NaN
1,1,328,Guyana,056,Belgium,Import,Sea,560750,"560750 | Twine, cordage, rope",5607,"5607 | Twine, cordage, rope",56,"56 | Wadding, felt and no",2731,"Twine, cordage, rope and cables",Textiles and Wearing Apparel,27,TEX - Textiles,4954.592,NaN,4954.592,kg,1267.0,1267.0,1268.871011,6364.0,3954.405044,5017.629927,['Container'],['Container'],26.226294,26.180503,26.184978,26.184978,26.205498,26.205498,27.434038,48.621722,30.260623,28.094172,28.170470,28.876023,26.862741,47.617531,31.851214,29.175236,28.505722,29.796748,26.361686,51.780837,39.497034,34.542712,31.958547,31.057036,25.869914,55.055720,45.304534,42.454194,34.215737,33.285370,25.387425,55.791260,53.303612,53.761921,37.093414,34.965209,24.895264,54.644530,52.366201,57.374868,42.439794,35.606067,24.431613,48.744246,50.804480,53.237793,39.908733,36.026771,23.958836,44.775756,45.515728,56.774996,46.904113,36.265343,2731,NaN


## 3.3 Inspect the output

In [10]:
impact_im_cpc_food = impact_im[(~impact_im.food_group.isna())]
impact_im_cpc_food_g = impact_im_cpc_food.groupby("food_group").agg({"HS6_desc":"count", "cifvalue":"sum", "vol_kg_vtw":"sum"}).reset_index().sort_values(by="cifvalue", ascending=False).round(2).rename(columns={"HS6_desc":"N_com"}).reset_index(drop=True)
impact_im_cpc_food_g["pct_val"] = round(100 * impact_im_cpc_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_cpc_food_g["pct_vol"] = round(100 * impact_im_cpc_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Proportion of imports identified as food security-relevant:\n")
print("Value: ${0}m of ${1}m ({2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / 1e6, 1),
    round(impact_im.cifvalue.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))
print("Volume: {0} of {1} million kilograms ({2}%)".format(
    round(impact_im_cpc_food.vol_kg_vtw.sum() / 1e6, 1),
    round(impact_im.vol_kg_vtw.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.vol_kg_vtw.sum() / impact_im.vol_kg_vtw.sum(), 1)
))
print("Value-to-Weight: {0} from {1} US$ per kg (+{2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum(), 2),
    round(impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum(), 2),
    round(100 * ((impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum()) - (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum())) / (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum()), 1)
))
print("Average distance to market (volume-weighted): {0} from {1} nm (+{2}%)".format(
    round((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000), 1),
    round((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000), 1),
    round(100 * (((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000)) - ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000))) / \
          ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000)), 1)
))
print("\n{0}\n".format(impact_im_cpc_food_g.head(15).round(1)))

impact_im_cpc_food_g.to_csv(output_dir + "{0}/food_total_usd_{1}.csv".format(iso_country, version), index=False)

Proportion of imports identified as food security-relevant:

Value: $354.9m of $9565.0m (3.7%)
Volume: 368.5 of 8006.2 million kilograms (4.6%)
Value-to-Weight: 0.96 from 1.19 US$ per kg (+-19.4%)
Average distance to market (volume-weighted): 3472.3 from 3647.7 nm (+-4.8%)

            food_group  N_com     cifvalue   vol_kg_vtw  pct_val  pct_vol
0    Processed Cereals    430  131088582.2   60580719.1      1.4      0.8
1                Dairy    123   69046102.3   22508797.7      0.7      0.3
2           Vegetables    212   60607965.2   87377335.1      0.6      1.1
3  Unprocessed Cereals     33   37744572.6  144357144.5      0.4      1.8
4        Fats and Oils     87   21514029.7   16184960.9      0.2      0.2
5      Sugar and Honey     71   19940361.4   27664457.3      0.2      0.3
6               Fruits     89   10228334.1    7999382.2      0.1      0.1
7        Meat and Prep     37    4774408.4    1804549.5      0.0      0.0



## 3.3 Derive results for 2029, 2041 and 2050

### 3.3.1 Total FEM costs by scenario (US$)

In [11]:
print("\nTotal FEM costs by scenario (US$)...\n")

d = {"food_group": list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["bau", "levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            round(100 * impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum(), 1) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,food_group,bau_29,levy_29,asis_29,brazil_29,liscr_29,direct_29,bau_41,levy_41,asis_41,brazil_41,liscr_41,direct_41,bau_50,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,77585448.8,137128078.3,85499467.8,79413478.7,79626563.7,81632906.6,71832988.0,157365864.6,150328150.1,151685370.6,104757060.1,98739713.0,67817622.7,126396226.8,128506119.9,160060704.2,132253335.4,102391434.5
1,Fats and Oils,47341739.3,81253345.1,51601948.2,48090353.2,48219532.1,49601223.4,44047877.1,92944917.9,88202180.6,89416686.9,62870033.2,59317708.3,41748676.8,75191104.7,77189913.2,94651403.5,77944562.6,61386221.2
2,Fruits,12179485.9,21581478.7,13432929.6,12473890.0,12507576.2,12819376.1,11271107.7,24765446.1,23656955.9,23858462.7,16462512.4,15522242.2,10637036.3,19882118.1,20209301.1,25200578.2,20809989.3,16099664.0
3,Meat and Prep,2813795.7,4986928.8,3103706.9,2881503.0,2889328.5,2961694.2,2603883.1,5722278.7,5467130.9,5514137.7,3804518.0,3586236.8,2457358.7,4592464.0,4668359.8,5823176.4,4810760.8,3719586.1
4,Processed Cereals,90784855.0,157265179.4,99393027.0,92650563.1,92872282.9,95268000.3,84360467.4,181044675.2,173003314.7,175124404.5,121506651.1,114310054.9,79876080.1,146235053.8,148483103.6,182939307.9,151606667.8,118363696.2
5,Sugar and Honey,37350216.6,62554163.9,40408828.6,37877903.5,37967594.9,39014146.6,34912093.3,72954717.9,69605171.5,70880620.6,49554330.8,46166782.2,33210169.3,59145826.8,59935209.9,72427389.4,60429979.6,47667983.4
6,Unprocessed Cereals,210127143.4,356142915.1,228138490.3,213389938.4,213955035.1,219808899.4,196005278.8,415450241.9,396171139.6,402710256.2,280688521.0,261292644.7,186147855.1,335194233.3,339836727.2,413348818.1,344454939.2,269997151.4
7,Vegetables,199049575.3,349321618.7,218857791.6,203548147.3,204068217.3,209239318.2,184532007.2,401241794.8,383411156.6,387302335.7,267868126.0,252372881.0,174397520.6,322995463.5,328181114.5,407201291.7,336820818.8,261574517.8


### 3.3.2 Total FEM costs by scenario (% cif)

In [12]:
print("\nTotal FEM costs by scenario (% cif)...\n")

d = {"food_group":list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            np.round(100 * (impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum() - impact_im[(impact_im.food_group == hsa4_codes_food_key)]["bau_usd_{1}".format(scenario, year)].sum()) / \
                     impact_im[(impact_im.food_group == hsa4_codes_food_key)].cifvalue.sum(), 2) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (% cif)...



,food_group,levy_29,asis_29,brazil_29,liscr_29,direct_29,levy_41,asis_41,brazil_41,liscr_41,direct_41,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,0.86,0.11,0.03,0.03,0.06,1.24,1.14,1.16,0.48,0.39,0.85,0.88,1.34,0.93,0.50
1,Fats and Oils,1.58,0.20,0.03,0.04,0.11,2.27,2.05,2.11,0.87,0.71,1.55,1.65,2.46,1.68,0.91
2,Fruits,0.92,0.12,0.03,0.03,0.06,1.32,1.21,1.23,0.51,0.42,0.90,0.94,1.42,0.99,0.53
3,Meat and Prep,0.46,0.06,0.01,0.02,0.03,0.65,0.60,0.61,0.25,0.21,0.45,0.46,0.70,0.49,0.26
4,Processed Cereals,0.51,0.07,0.01,0.02,0.03,0.74,0.68,0.69,0.28,0.23,0.51,0.52,0.79,0.55,0.29
5,Sugar and Honey,1.26,0.15,0.03,0.03,0.08,1.91,1.74,1.80,0.73,0.56,1.30,1.34,1.97,1.37,0.73
6,Unprocessed Cereals,3.87,0.48,0.09,0.10,0.26,5.81,5.30,5.48,2.24,1.73,3.95,4.07,6.02,4.19,2.22
7,Vegetables,2.48,0.33,0.07,0.08,0.17,3.58,3.28,3.35,1.38,1.12,2.45,2.54,3.84,2.68,1.44
